# 🏦 Loan Data — ML Pipeline (No Data Leakage + SMOTE)
> Rule: **Split first → fit pipeline on train only → evaluate on test**
> SMOTE is inside the pipeline so it only touches training folds — zero leakage.

## 1. Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, warnings
warnings.filterwarnings('ignore')

# ✅ Use imblearn Pipeline — supports sampler steps
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('loan_data.csv')

# Remove obvious outliers
df = df[df['person_age']     <= 100]
df = df[df['person_emp_exp'] <= 60]

print(df.shape)
df.head()

## 2. Data Visualization

In [ ]:
NUM_COLS = ['person_age','person_income','person_emp_exp','loan_amnt',
            'loan_int_rate','loan_percent_income','cb_person_cred_hist_length','credit_score']
CAT_COLS = ['person_gender','person_education','person_home_ownership',
            'loan_intent','previous_loan_defaults_on_file']

# Target distribution
vc = df['loan_status'].value_counts().rename({0: 'Rejected', 1: 'Approved'})
vc.plot(kind='bar', edgecolor='white', figsize=(5, 3))
plt.title('Loan Status Distribution (Before SMOTE)')
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

print('Class counts:')
print(vc)

In [ ]:
# Numeric distributions
df[NUM_COLS].hist(figsize=(18, 8), bins=30, color='steelblue', edgecolor='white')
plt.suptitle('Numeric Feature Distributions', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# Boxplots vs target
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, NUM_COLS):
    sns.boxplot(x='loan_status', y=col, data=df, ax=ax, palette='Set2')
    ax.set_title(col)
plt.tight_layout(); plt.show()

In [ ]:
# Categorical vs target
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, col in zip(axes.flat, CAT_COLS):
    sns.countplot(x=col, hue='loan_status', data=df, ax=ax,
                  order=df[col].value_counts().index, palette='Set1')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=30)
axes.flat[-1].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# Correlation heatmap
corr = df[NUM_COLS + ['loan_status']].corr()
plt.figure(figsize=(10, 7))
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)),
            annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout(); plt.show()

## 3. Train / Test Split — FIRST, before any fitting
> This is the key step that prevents data leakage.
> The pipeline will only **see** training data during `.fit()`.
> SMOTE will only oversample within training folds — test set is never touched.

In [ ]:
X = df[NUM_COLS + CAT_COLS]
y = df['loan_status']

# ✅ Split FIRST — no preprocessing has happened yet
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print()
print('Train class distribution (before SMOTE):')
print(y_train.value_counts().rename({0: 'Rejected', 1: 'Approved'}))
print()
print('Test class distribution (never resampled):')
print(y_test.value_counts().rename({0: 'Rejected', 1: 'Approved'}))

## 4. Build Pipeline (with SMOTE)

In [ ]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())          # learns mean/std from X_train only
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, NUM_COLS),
    ('cat', cat_pipe, CAT_COLS)
])

# ✅ SMOTE sits after preprocessing, before feature selection & classifier
# imblearn Pipeline ensures SMOTE only runs during fit(), not predict()
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('smote',        SMOTE(random_state=42)),
    ('selector',     SelectKBest(f_classif, k=8)),
    ('classifier',   RandomForestClassifier(n_estimators=100, random_state=42))
])

print('Pipeline steps:')
for name, step in pipeline.steps:
    print(f'  {name}: {step.__class__.__name__}')

## 5. Fit on Train Only — K-Fold CV (also on train only)

In [ ]:
# ✅ fit() only sees X_train
# Internally: preprocess → SMOTE oversample → select features → train classifier
pipeline.fit(X_train, y_train)

# ✅ K-Fold cross-validation on training set only
# SMOTE is applied inside each fold independently — no leakage
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X_train, y_train,
                             cv=kfold, scoring='accuracy', n_jobs=-1)

print(f'CV on train — Mean: {cv_scores.mean():.4f}  Std: {cv_scores.std():.4f}')
print(f'CV scores per fold: {np.round(cv_scores, 4)}')
print()

# Final evaluation on held-out test set
y_pred = pipeline.predict(X_test)
print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Rejected', 'Approved']))

In [ ]:
# CV per-fold chart + Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar([f'Fold {i+1}' for i in range(5)], cv_scores,
            color='steelblue', edgecolor='white')
axes[0].axhline(cv_scores.mean(), color='red', linestyle='--',
                label=f'Mean={cv_scores.mean():.4f}')
axes[0].set_ylim(0.6, 1.0)
axes[0].set_title('CV Accuracy per Fold (train set only)')
axes[0].legend()

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Rejected', 'Approved']).plot(
    ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Confusion Matrix (test set)')

plt.tight_layout(); plt.show()

## 6. Verify SMOTE Resampling (Optional Sanity Check)

In [ ]:
# Verify what SMOTE produced inside the pipeline on the full training set
X_preprocessed = pipeline.named_steps['preprocessor'].transform(X_train)
X_resampled, y_resampled = pipeline.named_steps['smote'].fit_resample(X_preprocessed, y_train)

print('Class distribution BEFORE SMOTE (train):')
before = pd.Series(y_train).value_counts().rename({0: 'Rejected', 1: 'Approved'})
print(before)

print()
print('Class distribution AFTER SMOTE (train):')
after = pd.Series(y_resampled).value_counts().rename({0: 'Rejected', 1: 'Approved'})
print(after)

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
before.plot(kind='bar', ax=axes[0], color=['#e74c3c','#2ecc71'], edgecolor='white')
axes[0].set_title('Before SMOTE')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

after.plot(kind='bar', ax=axes[1], color=['#e74c3c','#2ecc71'], edgecolor='white')
axes[1].set_title('After SMOTE')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.suptitle('SMOTE Effect on Training Set Class Balance')
plt.tight_layout(); plt.show()

## 7. Save Trained Pipeline for the App

In [ ]:
# ✅ Save the pipeline fitted on X_train only
# The saved pipeline includes preprocessor + SMOTE (inactive at predict time) + model
joblib.dump(pipeline, 'loan_pipeline.pkl')
print('Saved → loan_pipeline.pkl')
print('The app loads this file and calls .predict() directly.')
print('SMOTE is automatically skipped during prediction (imblearn behaviour).')